# Debug de Realizados: Fevereiro/2026

Este notebook lê todos os arquivos de Controle de Atendimentos das nutricionistas que estão na pasta `data/mensal/`, higieniza os cabeçalhos das colunas, filtra os dados pelo mês de fev/26 e conta a quantidade de "Realizado" conforme a regra de contabilização.

In [1]:
import pandas as pd
import glob
import os

# 1. Localizar os arquivos
DATA_DIR = os.path.join("data", "mensal")
FILES_E = glob.glob(os.path.join(DATA_DIR, "*Controle de atendimentos*.xlsx"))
print(f"Encontrados {len(FILES_E)} arquivos.")

def limpar_colunas(df):
    df.columns = df.columns.astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
    return df

Encontrados 10 arquivos.


In [2]:
# 2. Carregar e concatenar todas as planilhas
frames = []
col_status = 'Status atendimento (Realizado, Falta, Reagendou)'

for filepath in FILES_E:
    nome = os.path.basename(filepath)
    try:
        # As planilhas das nutris precisam pular as duas primeiras linhas de cabeçalho geral (skiprows=2)
        df = pd.read_excel(filepath, skiprows=2, sheet_name='Controle atendimentos')
        df = limpar_colunas(df)
        
        COLUNAS_E = ['Data', 'Nutri', 'ID caso', col_status]
        
        if all(c in df.columns for c in COLUNAS_E):
            df = df[COLUNAS_E].copy()
            df["Arquivo"] = nome
            frames.append(df)
        else:
            faltando = [c for c in COLUNAS_E if c not in df.columns]
            print(f"⚠️ Colunas faltando em {nome}: {faltando}")
    except Exception as e:
        print(f"❌ Erro ao ler {nome}: {e}")

df_all = pd.concat(frames, ignore_index=True)
df_all = df_all[df_all['Data'].notnull()].copy()
df_all['Data'] = pd.to_datetime(df_all['Data'], errors='coerce')
print(f"\nTotal de linhas agregadas antes do filtro de datas: {len(df_all)}")

c:\Users\vhcna\stranger_things\Freelas\Flua\flua_nutricao_mensal\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\vhcna\stranger_things\Freelas\Flua\flua_nutricao_mensal\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\vhcna\stranger_things\Freelas\Flua\flua_nutricao_mensal\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\vhcna\stranger_things\Freelas\Flua\flua_nutricao_mensal\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\vhcna\stranger_things\Freelas\Flua\flua_nutricao_mensal\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation exte


Total de linhas agregadas antes do filtro de datas: 5204


c:\Users\vhcna\stranger_things\Freelas\Flua\flua_nutricao_mensal\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [3]:
# 3. Filtrar para fevereiro de 2026
df_fev26 = df_all[(df_all['Data'].dt.year == 2026) & (df_all['Data'].dt.month == 2)].copy()

print(f"Total de agendamentos das Nutris em Fev/26: {len(df_fev26)}")

Total de agendamentos das Nutris em Fev/26: 373


In [4]:
# 4. Contar a quantidade de 'Realizado' por nutri em Fevereiro/2026
# Utilizando uma condição contains ignorando case para evitar erros de case (como Realizado vs realizado)
realizados_fev26 = df_fev26[df_fev26[col_status].astype(str).str.contains("Realizado", na=False, case=False)]

total_realizados = len(realizados_fev26)
print(f"✅ Total de atendimentos marcados como 'Realizado' em Fev/26: {total_realizados}\n")

print("=== Realizados por Nutri ===")
agrupamento_nutri = realizados_fev26.groupby('Nutri')['ID caso'].count().reset_index(name='Qtd Realizados')
display(agrupamento_nutri)

print("\n=== Realizados por Arquivo ===")
agrupamento_arquivo = realizados_fev26.groupby('Arquivo')['ID caso'].count().reset_index(name='Qtd Realizados')
display(agrupamento_arquivo)

✅ Total de atendimentos marcados como 'Realizado' em Fev/26: 202

=== Realizados por Nutri ===


,Nutri,Qtd Realizados
0,ANA LU SA R SZAJUBOK,20
1,ANDREIA DUTRA GONCALVES,28
2,BEATRIZ BOTEQUIO DE MORAES MACHADO,7
3,Cintia dos Santos Irineu,25
4,DANIELLE CASTELLANI,36
5,JOSE RIBEIRO GOUVEIA NETO,10
6,JULIANA RIBEIRO DE MELO,26
7,MANOELA MARIOTTO LANZARA,21
8,STEPHANY PATRICIA PEREZ MARTINEZ,29



=== Realizados por Arquivo ===


,Arquivo,Qtd Realizados
0,002 Controle de atendimentos_Bia.xlsx,7
1,02 Controle de atendimentos_AnaLu.xlsx,20
2,02 Controle de atendimentos_Andreia.xlsx,28
3,02 Controle de atendimentos_Cintia.xlsx,25
4,02 Controle de atendimentos_Danielle.xlsx,36
5,02 Controle de atendimentos_Juliana.xlsx,26
6,02 Controle de atendimentos_Manoela.xlsx,21
7,02 Controle de atendimentos_Neto.xlsx,10
8,02 Controle de atendimentos_Stephany.xlsx,29


In [9]:
# 5. Auditoria opcional de casos:
# Exibir todas as linhas de "Realizado" e ordená-las para descobrir se há duplicatas de ID caso
auditoria = realizados_fev26.sort_values(by=['Nutri', 'Data'])
display(auditoria)

# Validação de duplicados (mesmo paciente na mesma data por mesma nutri ou por planilhas diferentes)
duplicados =_ = realizados_fev26[realizados_fev26.duplicated(subset=['ID caso', 'Data'], keep=False)]
if not duplicados.empty:
    print("\n⚠️ Foram encontrados ID Casos duplicados em dadas iguais:")
    display(duplicados.sort_values('ID caso'))

,Data,Nutri,ID caso,"Status atendimento (Realizado, Falta, Reagendou)",Arquivo
1049874,2026-02-03,ANA LU SA R SZAJUBOK,507179.0,Realizado,02 Controle de atendimentos_AnaLu.xlsx
1049876,2026-02-03,ANA LU SA R SZAJUBOK,510761.0,Realizado,02 Controle de atendimentos_AnaLu.xlsx
1049877,2026-02-03,ANA LU SA R SZAJUBOK,505766.0,Realizado,02 Controle de atendimentos_AnaLu.xlsx
1049878,2026-02-03,ANA LU SA R SZAJUBOK,509017.0,Realizado,02 Controle de atendimentos_AnaLu.xlsx
1049881,2026-02-05,ANA LU SA R SZAJUBOK,505766.0,Realizado,02 Controle de atendimentos_AnaLu.xlsx
...,...,...,...,...,...
1073218,2026-02-25,STEPHANY PATRICIA PEREZ MARTINEZ,501903,Realizado,02 Controle de atendimentos_Stephany.xlsx
1073219,2026-02-25,STEPHANY PATRICIA PEREZ MARTINEZ,513529,Realizado,02 Controle de atendimentos_Stephany.xlsx
1073221,2026-02-25,STEPHANY PATRICIA PEREZ MARTINEZ,510113,Realizado,02 Controle de atendimentos_Stephany.xlsx
1073223,2026-02-25,STEPHANY PATRICIA PEREZ MARTINEZ,513297,Realizado,02 Controle de atendimentos_Stephany.xlsx


In [12]:
df_fev26

,Data,Nutri,ID caso,"Status atendimento (Realizado, Falta, Reagendou)",Arquivo
0,2026-02-23,BEATRIZ BOTEQUIO DE MORAES MACHADO,512880.0,Falta,002 Controle de atendimentos_Bia.xlsx
1,2026-02-23,BEATRIZ BOTEQUIO DE MORAES MACHADO,513087.0,Realizado,002 Controle de atendimentos_Bia.xlsx
2,2026-02-24,BEATRIZ BOTEQUIO DE MORAES MACHADO,513148.0,Realizado,002 Controle de atendimentos_Bia.xlsx
3,2026-02-25,BEATRIZ BOTEQUIO DE MORAES MACHADO,513455.0,Falta,002 Controle de atendimentos_Bia.xlsx
4,2026-02-25,BEATRIZ BOTEQUIO DE MORAES MACHADO,513082.0,Falta,002 Controle de atendimentos_Bia.xlsx
...,...,...,...,...,...
1073220,2026-02-25,STEPHANY PATRICIA PEREZ MARTINEZ,512108,Falta,02 Controle de atendimentos_Stephany.xlsx
1073221,2026-02-25,STEPHANY PATRICIA PEREZ MARTINEZ,510113,Realizado,02 Controle de atendimentos_Stephany.xlsx
1073222,2026-02-25,STEPHANY PATRICIA PEREZ MARTINEZ,513538,Falta,02 Controle de atendimentos_Stephany.xlsx
1073223,2026-02-25,STEPHANY PATRICIA PEREZ MARTINEZ,513297,Realizado,02 Controle de atendimentos_Stephany.xlsx


In [13]:
realizados_fev26

,Data,Nutri,ID caso,"Status atendimento (Realizado, Falta, Reagendou)",Arquivo
1,2026-02-23,BEATRIZ BOTEQUIO DE MORAES MACHADO,513087.0,Realizado,002 Controle de atendimentos_Bia.xlsx
2,2026-02-24,BEATRIZ BOTEQUIO DE MORAES MACHADO,513148.0,Realizado,002 Controle de atendimentos_Bia.xlsx
5,2026-02-25,BEATRIZ BOTEQUIO DE MORAES MACHADO,513573.0,Realizado,002 Controle de atendimentos_Bia.xlsx
6,2026-02-25,BEATRIZ BOTEQUIO DE MORAES MACHADO,513444.0,Realizado,002 Controle de atendimentos_Bia.xlsx
7,2026-02-26,BEATRIZ BOTEQUIO DE MORAES MACHADO,513467.0,Realizado,002 Controle de atendimentos_Bia.xlsx
...,...,...,...,...,...
1073218,2026-02-25,STEPHANY PATRICIA PEREZ MARTINEZ,501903,Realizado,02 Controle de atendimentos_Stephany.xlsx
1073219,2026-02-25,STEPHANY PATRICIA PEREZ MARTINEZ,513529,Realizado,02 Controle de atendimentos_Stephany.xlsx
1073221,2026-02-25,STEPHANY PATRICIA PEREZ MARTINEZ,510113,Realizado,02 Controle de atendimentos_Stephany.xlsx
1073223,2026-02-25,STEPHANY PATRICIA PEREZ MARTINEZ,513297,Realizado,02 Controle de atendimentos_Stephany.xlsx
